# Model Evaluation: Fine-tuned vs. Pre-trained


This notebook evaluates fine-tuned models and pre-trained models (without fine-tuning) on the mental health datasets. The goal is to compare model performance before and after fine-tuning, using the previously prepared test sets.

## Importing Libraries and Utilities

In [1]:
# Import system libraries and set up path for utility modules
import sys
import os

# Add the "src" directory to sys.path to import pipeline utility functions
sys.path.append(os.path.abspath(os.path.join(os.pardir, "src")))

In [2]:
# Import main libraries for data handling, models, and visualization
import pandas as pd
import torch
from transformers import (
    BertForSequenceClassification,
    BertTokenizer,
    RobertaForSequenceClassification,
    RobertaTokenizer,
)

In [3]:
# Import the main evaluation function
from evaluate import evaluate_model_on_test

## General Definitions

In [4]:
# List of models and tokenizers to evaluate (name, model class, tokenizer class)
MODELS = [
    ("google-bert/bert-base-uncased", BertForSequenceClassification, BertTokenizer),
    ("mental/mental-roberta-base", RobertaForSequenceClassification, RobertaTokenizer),
]

In [5]:
# Global parameter definitions for model evaluation
BATCH_SIZE = 16
MAX_LENGTH = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
# Initialize global lists for results and detailed reports
RESULTS_SUMMARY = []

In [7]:
# Print evaluation metrics in a formatted way
def print_evaluation_metrics(metrics, model_name, prefix="", class_names=None):
    separator = "=" * 80
    sub_separator = "-" * 80

    print(separator)
    print(f"Model: ({prefix}) {model_name}")

    print(sub_separator)
    print(f"Accuracy : {metrics.get('accuracy', 0):.10f}")
    print(f"Precision: {metrics.get('precision', 0):.10f}")
    print(f"Recall   : {metrics.get('recall', 0):.10f}")
    print(f"F1-score : {metrics.get('f1', 0):.10f}")

    cm = metrics.get("confusion_matrix")
    if cm is not None:
        print(sub_separator)
        print("Confusion Matrix:")
        if class_names is not None:
            cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
            cm_df.index.name = "True"
            cm_df.columns.name = "Predicted"
        else:
            cm_df = pd.DataFrame(cm)
        print(cm_df)

    cr = metrics.get("classification_report")
    if cr is not None:
        print(sub_separator)
        print("Detailed classification report:")
        print(cr)

    print(separator)

In [8]:
# Store results in the global list for later visualization
def add_result(dataset, model, tuning, metrics):
    RESULTS_SUMMARY.append({
        "Dataset": dataset,
        "Model": model,
        "Fine-tuning": tuning,
        "Accuracy": metrics["accuracy"],
        "Precision": metrics["precision"],
        "Recall": metrics["recall"],
        "F1-score": metrics["f1"],
    })

## Evaluation by Dataset


Each dataset will be evaluated separately, comparing the performance of fine-tuned and pre-trained models.

### Suicide and Depression Detection Dataset

In [9]:
# Load test data for Suicide and Depression Detection
DATA_DIR = "../data/processed/suicide-and-depression-detection"
TEST_DF = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

In [10]:
# Base directory for models
BASE_PATH = "../models/suicide-and-depression-detection"

In [11]:
# Evaluate pre-trained models (without fine-tuning) - Suicide and Depression Detection
for model_name, model_cls, tokenizer_cls in MODELS:
    tokenizer = tokenizer_cls.from_pretrained(model_name)
    model = model_cls.from_pretrained(
        model_name,
        num_labels=len(TEST_DF["label"].unique()),
    )
    model.to(DEVICE)
    metrics_base = evaluate_model_on_test(
        model,
        tokenizer,
        TEST_DF,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        device=DEVICE,
    )

    print_evaluation_metrics(metrics_base, model_name, prefix="Pre-trained", class_names=TEST_DF["label"].unique().tolist())
    add_result("Suicide and Depression Detection", model_name, "Pre-trained", metrics_base)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: (Pre-trained) google-bert/bert-base-uncased
--------------------------------------------------------------------------------
Accuracy : 0.5000000000
Precision: 0.2500000000
Recall   : 0.5000000000
F1-score : 0.3333333333
--------------------------------------------------------------------------------
Confusion Matrix:
Predicted    suicide  not_suicide
True                             
suicide            0        11558
not_suicide        0        11558
--------------------------------------------------------------------------------
Detailed classification report:
              precision    recall  f1-score   support

 not_suicide       0.00      0.00      0.00     11558
     suicide       0.50      1.00      0.67     11558

    accuracy                           0.50     23116
   macro avg       0.25      0.50      0.33     23116
weighted avg       0.25      0.50      0.33     23116



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at mental/mental-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: (Pre-trained) mental/mental-roberta-base
--------------------------------------------------------------------------------
Accuracy : 0.5000000000
Precision: 0.2500000000
Recall   : 0.5000000000
F1-score : 0.3333333333
--------------------------------------------------------------------------------
Confusion Matrix:
Predicted    suicide  not_suicide
True                             
suicide            0        11558
not_suicide        0        11558
--------------------------------------------------------------------------------
Detailed classification report:
              precision    recall  f1-score   support

 not_suicide       0.00      0.00      0.00     11558
     suicide       0.50      1.00      0.67     11558

    accuracy                           0.50     23116
   macro avg       0.25      0.50      0.33     23116
weighted avg       0.25      0.50      0.33     23116



In [12]:
# Evaluate fine-tuned models - Suicide and Depression Detection
for model_name, model_cls, tokenizer_cls in MODELS:
    save_dir = f"{BASE_PATH}/{model_name.replace('/', '_')}_hf"
    model_dir = f"{save_dir}/model"
    tokenizer_dir = f"{save_dir}/tokenizer"
    tokenizer = tokenizer_cls.from_pretrained(tokenizer_dir)
    model = model_cls.from_pretrained(
        model_dir, num_labels=len(TEST_DF["label"].unique())
    )
    model.to(DEVICE)
    metrics_ft = evaluate_model_on_test(
        model,
        tokenizer,
        TEST_DF,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        device=DEVICE,
    )

    print_evaluation_metrics(metrics_ft, model_name, prefix="Fine-tuned", class_names=TEST_DF["label"].unique().tolist())
    add_result("Suicide and Depression Detection", model_name, "Fine-tuned", metrics_ft)

Model: (Fine-tuned) google-bert/bert-base-uncased
--------------------------------------------------------------------------------
Accuracy : 0.9809223049
Precision: 0.9809242093
Recall   : 0.9809223049
F1-score : 0.9809222860
--------------------------------------------------------------------------------
Confusion Matrix:
Predicted    suicide  not_suicide
True                             
suicide        11326          232
not_suicide      209        11349
--------------------------------------------------------------------------------
Detailed classification report:
              precision    recall  f1-score   support

 not_suicide       0.98      0.98      0.98     11558
     suicide       0.98      0.98      0.98     11558

    accuracy                           0.98     23116
   macro avg       0.98      0.98      0.98     23116
weighted avg       0.98      0.98      0.98     23116

Model: (Fine-tuned) mental/mental-roberta-base
---------------------------------------------------

### Sentiment Analysis for Mental Health Dataset

In [13]:
# Load test data for Sentiment Analysis for Mental Health
DATA_DIR = "../data/processed/sentiment-analysis-for-mental-health"
TEST_DF = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

In [14]:
# Base directory for models
BASE_PATH = "../models/sentiment-analysis-for-mental-health"

In [15]:
# Evaluate pre-trained models (without fine-tuning) - Sentiment Analysis for Mental Health
for model_name, model_cls, tokenizer_cls in MODELS:
    tokenizer = tokenizer_cls.from_pretrained(model_name)
    model = model_cls.from_pretrained(
        model_name,
        num_labels=len(TEST_DF["label"].unique()),
    )
    model.to(DEVICE)
    metrics_base = evaluate_model_on_test(
        model,
        tokenizer,
        TEST_DF,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        device=DEVICE,
    )

    print_evaluation_metrics(metrics_base, model_name, prefix="Pre-trained", class_names=TEST_DF["label"].unique().tolist())
    add_result("Sentiment Analysis for Mental Health", model_name, "Pre-trained", metrics_base)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: (Pre-trained) google-bert/bert-base-uncased
--------------------------------------------------------------------------------
Accuracy : 0.0194041552
Precision: 0.1429962349
Recall   : 0.0194041552
F1-score : 0.0032865382
--------------------------------------------------------------------------------
Confusion Matrix:
Predicted             normal  suicide  stress  anxiety  depression  bipolar  \
True                                                                          
normal                     5        0       0        0         354        1   
suicide                    2        0       0        0         246        2   
stress                    25        0       2        0        1475        2   
anxiety                  235        0       0        0        1354        2   
depression                 0        0       0        0          90        0   
bipolar                    2        0       0        0         226        0   
personality_disorder      24        0    

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at mental/mental-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: (Pre-trained) mental/mental-roberta-base
--------------------------------------------------------------------------------
Accuracy : 0.3371226970
Precision: 0.2518153138
Recall   : 0.3371226970
F1-score : 0.2403456127
--------------------------------------------------------------------------------
Confusion Matrix:
Predicted             normal  suicide  stress  anxiety  depression  bipolar  \
True                                                                          
normal                     0        0     304       57           0        0   
suicide                    0        0     202       48           0        0   
stress                     0        0    1317      165          27        0   
anxiety                    0        0    1197      403           0        0   
depression                 0        0      77       13           0        0   
bipolar                    0        0     192       37           0        0   
personality_disorder       0        0    100

In [16]:
# Evaluate fine-tuned models - Sentiment Analysis for Mental Health
for model_name, model_cls, tokenizer_cls in MODELS:
    save_dir = f"{BASE_PATH}/{model_name.replace('/', '_')}_hf"
    model_dir = f"{save_dir}/model"
    tokenizer_dir = f"{save_dir}/tokenizer"
    tokenizer = tokenizer_cls.from_pretrained(tokenizer_dir)
    model = model_cls.from_pretrained(
        model_dir, num_labels=len(TEST_DF["label"].unique())
    )
    model.to(DEVICE)
    metrics_ft = evaluate_model_on_test(
        model,
        tokenizer,
        TEST_DF,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        device=DEVICE,
    )

    print_evaluation_metrics(metrics_ft, model_name, prefix="Fine-tuned", class_names=TEST_DF["label"].unique().tolist())
    add_result("Sentiment Analysis for Mental Health", model_name, "Fine-tuned", metrics_ft)

Model: (Fine-tuned) google-bert/bert-base-uncased
--------------------------------------------------------------------------------
Accuracy : 0.8382987064
Precision: 0.8378009742
Recall   : 0.8382987064
F1-score : 0.8377556689
--------------------------------------------------------------------------------
Confusion Matrix:
Predicted             normal  suicide  stress  anxiety  depression  bipolar  \
True                                                                          
normal                   321       10       7        6           5       12   
suicide                   10      209      15        3           5        8   
stress                     6       18    1158       23           7        7   
anxiety                    3        0      11     1552           1       29   
depression                 3        8      14        5          53        5   
bipolar                   15        0       6       23           4      180   
personality_disorder       1        0     

### Sentimental Analysis for Tweets Dataset

In [17]:
# Load test data for Sentimental Analysis for Tweets
DATA_DIR = "../data/processed/sentimental-analysis-for-tweets"
TEST_DF = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

In [18]:
# Base directory for models
BASE_PATH = "../models/sentimental-analysis-for-tweets"

In [19]:
# Evaluate pre-trained models (without fine-tuning) - Sentimental Analysis for Tweets
for model_name, model_cls, tokenizer_cls in MODELS:
    tokenizer = tokenizer_cls.from_pretrained(model_name)
    model = model_cls.from_pretrained(
        model_name,
        num_labels=len(TEST_DF["label"].unique()),
    )
    model.to(DEVICE)
    metrics_base = evaluate_model_on_test(
        model,
        tokenizer,
        TEST_DF,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        device=DEVICE,
    )

    print_evaluation_metrics(metrics_base, model_name, prefix="Pre-trained", class_names=TEST_DF["label"].unique().tolist())
    add_result("Sentimental Analysis for Tweets", model_name, "Pre-trained", metrics_base)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: (Pre-trained) google-bert/bert-base-uncased
--------------------------------------------------------------------------------
Accuracy : 0.5203619910
Precision: 0.6031960154
Recall   : 0.5203619910
F1-score : 0.3999487705
--------------------------------------------------------------------------------
Confusion Matrix:
Predicted       depression  not_depression
True                                      
depression             214               7
not_depression         205              16
--------------------------------------------------------------------------------
Detailed classification report:
                precision    recall  f1-score   support

    depression       0.51      0.97      0.67       221
not_depression       0.70      0.07      0.13       221

      accuracy                           0.52       442
     macro avg       0.60      0.52      0.40       442
  weighted avg       0.60      0.52      0.40       442



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at mental/mental-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: (Pre-trained) mental/mental-roberta-base
--------------------------------------------------------------------------------
Accuracy : 0.5000000000
Precision: 0.2500000000
Recall   : 0.5000000000
F1-score : 0.3333333333
--------------------------------------------------------------------------------
Confusion Matrix:
Predicted       depression  not_depression
True                                      
depression               0             221
not_depression           0             221
--------------------------------------------------------------------------------
Detailed classification report:
                precision    recall  f1-score   support

    depression       0.00      0.00      0.00       221
not_depression       0.50      1.00      0.67       221

      accuracy                           0.50       442
     macro avg       0.25      0.50      0.33       442
  weighted avg       0.25      0.50      0.33       442



In [20]:
# Evaluate fine-tuned models - Sentimental Analysis for Tweets
for model_name, model_cls, tokenizer_cls in MODELS:
    save_dir = f"{BASE_PATH}/{model_name.replace('/', '_')}_hf"
    model_dir = f"{save_dir}/model"
    tokenizer_dir = f"{save_dir}/tokenizer"
    tokenizer = tokenizer_cls.from_pretrained(tokenizer_dir)
    model = model_cls.from_pretrained(
        model_dir, num_labels=len(TEST_DF["label"].unique())
    )
    model.to(DEVICE)
    metrics_ft = evaluate_model_on_test(
        model,
        tokenizer,
        TEST_DF,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        device=DEVICE,
    )

    print_evaluation_metrics(metrics_ft, model_name, prefix="Fine-tuned", class_names=TEST_DF["label"].unique().tolist())
    add_result("Sentimental Analysis for Tweets", model_name, "Fine-tuned", metrics_ft)

Model: (Fine-tuned) google-bert/bert-base-uncased
--------------------------------------------------------------------------------
Accuracy : 0.9954751131
Precision: 0.9955156951
Recall   : 0.9954751131
F1-score : 0.9954750205
--------------------------------------------------------------------------------
Confusion Matrix:
Predicted       depression  not_depression
True                                      
depression             219               2
not_depression           0             221
--------------------------------------------------------------------------------
Detailed classification report:
                precision    recall  f1-score   support

    depression       1.00      0.99      1.00       221
not_depression       0.99      1.00      1.00       221

      accuracy                           1.00       442
     macro avg       1.00      1.00      1.00       442
  weighted avg       1.00      1.00      1.00       442

Model: (Fine-tuned) mental/mental-roberta-base
---

### Mental Health Corpus Dataset

In [21]:
# Load test data for Mental Health Corpus
DATA_DIR = "../data/processed/mental-health-corpus"
TEST_DF = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

In [22]:
# Base directory for models
BASE_PATH = "../models/mental-health-corpus"

In [23]:
# Evaluate pre-trained models (without fine-tuning) - Mental Health Corpus
for model_name, model_cls, tokenizer_cls in MODELS:
    tokenizer = tokenizer_cls.from_pretrained(model_name)
    model = model_cls.from_pretrained(
        model_name,
        num_labels=len(TEST_DF["label"].unique()),
    )
    model.to(DEVICE)
    metrics_base = evaluate_model_on_test(
        model,
        tokenizer,
        TEST_DF,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        device=DEVICE,
    )

    print_evaluation_metrics(metrics_base, model_name, prefix="Pre-trained", class_names=TEST_DF["label"].unique().tolist())
    add_result("Mental Health Corpus", model_name, "Pre-trained", metrics_base)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: (Pre-trained) google-bert/bert-base-uncased
--------------------------------------------------------------------------------
Accuracy : 0.5034532897
Precision: 0.5789806081
Recall   : 0.5034532897
F1-score : 0.3479695788
--------------------------------------------------------------------------------
Confusion Matrix:
Predicted      not_poisonous  poisonous
True                                   
not_poisonous             21       1355
poisonous                 11       1364
--------------------------------------------------------------------------------
Detailed classification report:
               precision    recall  f1-score   support

not_poisonous       0.66      0.02      0.03      1376
    poisonous       0.50      0.99      0.67      1375

     accuracy                           0.50      2751
    macro avg       0.58      0.50      0.35      2751
 weighted avg       0.58      0.50      0.35      2751



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at mental/mental-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: (Pre-trained) mental/mental-roberta-base
--------------------------------------------------------------------------------
Accuracy : 0.4998182479
Precision: 0.2498182809
Recall   : 0.4998182479
F1-score : 0.3331314061
--------------------------------------------------------------------------------
Confusion Matrix:
Predicted      not_poisonous  poisonous
True                                   
not_poisonous              0       1376
poisonous                  0       1375
--------------------------------------------------------------------------------
Detailed classification report:
               precision    recall  f1-score   support

not_poisonous       0.00      0.00      0.00      1376
    poisonous       0.50      1.00      0.67      1375

     accuracy                           0.50      2751
    macro avg       0.25      0.50      0.33      2751
 weighted avg       0.25      0.50      0.33      2751



In [24]:
# Evaluate fine-tuned models - Mental Health Corpus
for model_name, model_cls, tokenizer_cls in MODELS:
    save_dir = f"{BASE_PATH}/{model_name.replace('/', '_')}_hf"
    model_dir = f"{save_dir}/model"
    tokenizer_dir = f"{save_dir}/tokenizer"
    tokenizer = tokenizer_cls.from_pretrained(tokenizer_dir)
    model = model_cls.from_pretrained(
        model_dir, num_labels=len(TEST_DF["label"].unique())
    )
    model.to(DEVICE)
    metrics_ft = evaluate_model_on_test(
        model,
        tokenizer,
        TEST_DF,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        device=DEVICE,
    )

    print_evaluation_metrics(metrics_ft, model_name, prefix="Fine-tuned", class_names=TEST_DF["label"].unique().tolist())
    add_result("Mental Health Corpus", model_name, "Fine-tuned", metrics_ft)

Model: (Fine-tuned) google-bert/bert-base-uncased
--------------------------------------------------------------------------------
Accuracy : 0.9665576154
Precision: 0.9665733274
Recall   : 0.9665576154
F1-score : 0.9665572972
--------------------------------------------------------------------------------
Confusion Matrix:
Predicted      not_poisonous  poisonous
True                                   
not_poisonous           1334         42
poisonous                 50       1325
--------------------------------------------------------------------------------
Detailed classification report:
               precision    recall  f1-score   support

not_poisonous       0.96      0.97      0.97      1376
    poisonous       0.97      0.96      0.97      1375

     accuracy                           0.97      2751
    macro avg       0.97      0.97      0.97      2751
 weighted avg       0.97      0.97      0.97      2751

Model: (Fine-tuned) mental/mental-roberta-base
---------------------

## Comparative Visualization of Results


This section presents visualizations to facilitate comparison between the evaluated models and datasets.

In [25]:
# Create summary DataFrame after running all experiments
df_results = pd.DataFrame(RESULTS_SUMMARY)

In [26]:
# Display summary table of final metrics
if not df_results.empty:
    display(df_results.round(6))

,Dataset,Model,Fine-tuning,Accuracy,Precision,Recall,F1-score
0,Suicide and Depression Detection,google-bert/bert-base-uncased,Pre-trained,0.500000,0.250000,0.500000,0.333333
1,Suicide and Depression Detection,mental/mental-roberta-base,Pre-trained,0.500000,0.250000,0.500000,0.333333
2,Suicide and Depression Detection,google-bert/bert-base-uncased,Fine-tuned,0.980922,0.980924,0.980922,0.980922
3,Suicide and Depression Detection,mental/mental-roberta-base,Fine-tuned,0.991824,0.991836,0.991824,0.991824
4,Sentiment Analysis for Mental Health,google-bert/bert-base-uncased,Pre-trained,0.019404,0.142996,0.019404,0.003287
5,Sentiment Analysis for Mental Health,mental/mental-roberta-base,Pre-trained,0.337123,0.251815,0.337123,0.240346
6,Sentiment Analysis for Mental Health,google-bert/bert-base-uncased,Fine-tuned,0.838299,0.837801,0.838299,0.837756
7,Sentiment Analysis for Mental Health,mental/mental-roberta-base,Fine-tuned,0.845747,0.846793,0.845747,0.845967
8,Sentimental Analysis for Tweets,google-bert/bert-base-uncased,Pre-trained,0.520362,0.603196,0.520362,0.399949
9,Sentimental Analysis for Tweets,mental/mental-roberta-base,Pre-trained,0.500000,0.250000,0.500000,0.333333


## Final Remarks


- The pre-trained models (without fine-tuning) showed unsatisfactory performance on the evaluated datasets. This was expected, as their final classification layers were not specifically adapted to the mental health tasks considered.
- After fine-tuning, all models showed significant improvement, reaching metrics above 95% for accuracy, precision, recall, and F1-score on most datasets. This highlights the importance of fine-tuning for specific tasks.
- In unbalanced datasets, it was observed that majority classes influenced overall performance, resulting in higher metrics for these classes and lower performance for minority classes. This result reinforces the need for strategies to handle imbalance.
- Overall, fine-tuning was essential to adapt the models to the particularities of mental health data, making them suitable for real-world applications in this domain.